# Loan Payback Classifier

## EDA

In [37]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    classification_report, 
    confusion_matrix, 
    roc_auc_score, 
    roc_curve, 
    accuracy_score,
    f1_score
)
from sklearn.linear_model import LogisticRegression
from imblearn.over_sampling import SMOTE

from xgboost import XGBClassifier
import lightgbm as lgb
from catboost import CatBoostClassifier

In [5]:
train_path = r"../data/loan_payback/playground-series-s5e11/train.csv"
test_path = r"../data/loan_payback/playground-series-s5e11/test.csv"

In [6]:
get_input = lambda x: pd.read_csv(x)

In [ ]:
def evaluate_model(y_test, y_pred, y_prob):
# --- 5. Evaluation metrics
    print("✅ Accuracy:", accuracy_score(y_test, y_pred))
    print("\n📋 Classification Report:\n", classification_report(y_test, y_pred))
    print("🔢 Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("\n🏅 ROC AUC Score:", roc_auc_score(y_test, y_prob))

    # --- 6. Plot ROC Curve
    fpr, tpr, thresholds = roc_curve(y_test, y_prob)
    plt.plot(fpr, tpr, label=f"ROC curve (AUC = {roc_auc_score(y_test, y_prob):.2f})")
    plt.plot([0, 1], [0, 1], 'k--')  # diagonal line
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC Curve")
    plt.legend()
    plt.show()

In [ ]:

# Identify feature types
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object', 'category']).columns

# Preprocessing
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

# Model with class weights
log_reg = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

# Fit
log_reg.fit(X_train, y_train)

# Predict
y_pred = log_reg.predict(X_test)
y_proba = log_reg.predict_proba(X_test)[:, 1]


In [ ]:
# from imblearn.over_sampling import SMOTE
# from imblearn.pipeline import Pipeline as ImbPipeline

# smote = SMOTE(random_state=42)

# balanced_model = ImbPipeline(steps=[
#     ('preprocessor', preprocessor),
#     ('smote', smote),
#     ('classifier', LogisticRegression(max_iter=1000))
# ])

# balanced_model.fit(X_train, y_train)
# y_pred_smote = balanced_model.predict(X_test)
# y_proba_smote = balanced_model.predict_proba(X_test)[:, 1]

# print("After SMOTE ROC-AUC:", roc_auc_score(y_test, y_proba_smote))


In [224]:
def classifier_splits(X, y):
# """Split for classification unbalanced"""
   return train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [7]:
raw_df = get_input(train_path)

In [8]:
raw_df.head()

,id,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,gender,marital_status,education_level,employment_status,loan_purpose,grade_subgrade,loan_paid_back
0,0,29367.99,0.084,736,2528.42,13.67,Female,Single,High School,Self-employed,Other,C3,1.0
1,1,22108.02,0.166,636,4593.10,12.92,Male,Married,Master's,Employed,Debt consolidation,D3,0.0
2,2,49566.20,0.097,694,17005.15,9.76,Male,Single,High School,Employed,Debt consolidation,C5,1.0
3,3,46858.25,0.065,533,4682.48,16.10,Female,Single,High School,Employed,Debt consolidation,F1,1.0
4,4,25496.70,0.053,665,12184.43,10.21,Male,Married,High School,Employed,Other,D1,1.0


In [9]:
raw_df.describe()

,id,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,loan_paid_back
count,593994.000000,593994.000000,593994.000000,593994.000000,593994.000000,593994.000000,593994.000000
mean,296996.500000,48212.202976,0.120696,680.916009,15020.297629,12.356345,0.798820
std,171471.442236,26711.942078,0.068573,55.424956,6926.530568,2.008959,0.400883
min,0.000000,6002.430000,0.011000,395.000000,500.090000,3.200000,0.000000
25%,148498.250000,27934.400000,0.072000,646.000000,10279.620000,10.990000,1.000000
50%,296996.500000,46557.680000,0.096000,682.000000,15000.220000,12.370000,1.000000
75%,445494.750000,60981.320000,0.156000,719.000000,18858.580000,13.680000,1.000000
max,593993.000000,393381.740000,0.627000,849.000000,48959.950000,20.990000,1.000000


In [10]:
cat = [i for i in raw_df.columns if raw_df[i].dtype == "object"]
target = "loan_paid_back"
num = [i for i in raw_df.columns if i not in (cat + [target, "id"])]

In [11]:
print("Categorical Features: ", cat)
print("Numerical Features: ", num)
print("Target Feature: ", [target])

Categorical Features:  ['gender', 'marital_status', 'education_level', 'employment_status', 'loan_purpose', 'grade_subgrade']
Numerical Features:  ['annual_income', 'debt_to_income_ratio', 'credit_score', 'loan_amount', 'interest_rate']
Target Feature:  ['loan_paid_back']


In [212]:
for i in cat:
    print("*" * 40)
    print(f"\nValue counts for {i}L")
    print(raw_df[i].value_counts())


****************************************

Value counts for genderL
gender
Female    306175
Male      284091
Other       3728
Name: count, dtype: int64
****************************************

Value counts for marital_statusL
marital_status
Single      288843
Married     277239
Divorced     21312
Widowed       6600
Name: count, dtype: int64
****************************************

Value counts for education_levelL
education_level
Bachelor's     279606
High School    183592
Master's        93097
Other           26677
PhD             11022
Name: count, dtype: int64
****************************************

Value counts for employment_statusL
employment_status
Employed         450645
Unemployed        62485
Self-employed     52480
Retired           16453
Student           11931
Name: count, dtype: int64
****************************************

Value counts for loan_purposeL
loan_purpose
Debt consolidation    324695
Other                  63874
Car                    58108
Home          

In [ ]:
raw_df[(raw_df["employment_status"] == 'Student') & (raw_df["loan_paid_back"] == 0)].gender.value_counts()

gender
Female    245463
Male      226066
Other       2965
Name: count, dtype: int64

In [208]:
raw_df.columns

Index(['id', 'annual_income', 'debt_to_income_ratio', 'credit_score',
       'loan_amount', 'interest_rate', 'gender', 'marital_status',
       'education_level', 'employment_status', 'loan_purpose',
       'grade_subgrade', 'loan_paid_back'],
      dtype='object')

In [220]:
raw_df.loan_paid_back.value_counts()

loan_paid_back
1.0    474494
0.0    119500
Name: count, dtype: int64

In [39]:
# financial feature engineering
def domain_featuring(raw_df):
    featured_df = raw_df.copy()
    featured_df['credit_utilization'] = featured_df['loan_amount'] / featured_df['annual_income']
    featured_df['interest_burden'] = featured_df['interest_rate'] * featured_df['loan_amount'] / 100
    featured_df['is_employed_flag'] = featured_df['employment_status'].apply(lambda x: 0 if x in ['Unemployed', 'Student', 'Retired'] else 1)
    featured_df['debt_intrest'] = featured_df['debt_to_income_ratio'] * featured_df['interest_rate'] / 100
    featured_df['credit_debt_interaction'] = featured_df['credit_score'] * featured_df['debt_to_income_ratio']

    # Example: Tagging high risk
    featured_df['high_risk_combo'] = (
        (featured_df['employment_status'].isin(['Unemployed', 'Student'])) &
        (featured_df['education_level'].isin(['High School', 'Other'])) &
        (featured_df['marital_status'].isin(['Single', 'Divorced', 'Widowed'])) &
        (featured_df['loan_purpose'].isin(['Medical', 'Education', 'Vacation', 'Other'])) &
        (featured_df['grade_subgrade'].str[0].isin(['D', 'E', 'F']))
    )
    return featured_df


In [ ]:
(featured_df['grade_subgrade'].str[0].isin(['D', 'E', 'F']))

grade_subgrade
False    593994
Name: count, dtype: int64

In [22]:
featured_df.high_risk_combo.value_counts()

high_risk_combo
False    592306
True       1688
Name: count, dtype: int64

In [34]:
print(featured_df[(featured_df.high_risk_combo == True)]['loan_paid_back'].value_counts())

loan_paid_back
0.0    1649
1.0      39
Name: count, dtype: int64


In [36]:
featured_df[(featured_df.high_risk_combo == False) & (featured_df.loan_paid_back == 0)].head(50)

,id,annual_income,debt_to_income_ratio,credit_score,loan_amount,interest_rate,gender,marital_status,education_level,employment_status,loan_purpose,grade_subgrade,loan_paid_back,credit_utilization,interest_burden,is_employed_flag,debt_intrest,credit_debt_interaction,high_risk_combo
1,1,22108.02,0.166,636,4593.10,12.92,Male,Married,Master's,Employed,Debt consolidation,D3,0.0,0.207757,593.428520,1,0.021447,105.576,False
14,14,22790.15,0.083,665,3363.68,13.44,Male,Married,Bachelor's,Unemployed,Debt consolidation,D1,0.0,0.147594,452.078592,0,0.011155,55.195,False
17,17,35268.70,0.089,576,23642.37,12.60,Male,Married,Bachelor's,Unemployed,Debt consolidation,F5,0.0,0.670350,2978.938620,0,0.011214,51.264,False
22,22,135416.72,0.160,661,12404.14,13.30,Male,Single,High School,Unemployed,Debt consolidation,D1,0.0,0.091600,1649.750620,0,0.021280,105.760,False
26,26,24001.73,0.284,669,13863.79,10.88,Female,Married,Master's,Employed,Debt consolidation,D4,0.0,0.577616,1508.380352,1,0.030899,189.996,False
27,27,26381.68,0.079,718,19364.62,9.22,Female,Single,Bachelor's,Unemployed,Car,C4,0.0,0.734018,1785.417964,0,0.007284,56.722,False
29,29,52238.72,0.136,642,12800.86,13.35,Female,Married,Bachelor's,Unemployed,Car,D2,0.0,0.245045,1708.914810,0,0.018156,87.312,False
34,34,17420.25,0.271,662,9702.64,10.93,Male,Single,Master's,Employed,Debt consolidation,D5,0.0,0.556975,1060.498552,1,0.029620,179.402,False
35,35,98445.89,0.166,704,2992.43,14.10,Female,Single,Bachelor's,Employed,Business,C2,0.0,0.030397,421.932630,1,0.023406,116.864,False
36,36,48602.37,0.091,531,13379.10,15.67,Male,Married,Bachelor's,Unemployed,Debt consolidation,F4,0.0,0.275277,2096.504970,0,0.014260,48.321,False


In [45]:
def model_evaluation(model, data):
    
    if isinstance(model, LogisticRegression):
        cat = [i for i in data.columns if raw_df[i].dtype == "object"]
        data = pd.get_dummies(data.copy(), columns=cat, drop_first=True)


    cols = [i for i in data.columns if i not in [target, 'id']]
    X, y = data[cols], data[target]
        
    # Initialize StratifiedKFold (5 folds)
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    fold = 1
    accuracies, f1s, aucs = [], [], []

    for train_index, test_index in skf.split(X, y):
        print(f"\n🔹 Fold {fold}")
        
        # Split the data
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y.iloc[train_index], y.iloc[test_index]
        
        # Train the model
        model.fit(X_train, y_train)
        
        # Predict
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)[:, 1]  # needed for ROC AUC
        
        # Evaluate
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_proba)
        
        accuracies.append(acc)
        f1s.append(f1)
        aucs.append(auc)
        
        print(f"Accuracy: {acc:.4f}")
        print(f"F1 Score: {f1:.4f}")
        print(f"ROC AUC: {auc:.4f}")
        print("Confusion Matrix:")
        print(confusion_matrix(y_test, y_pred))
        print(classification_report(y_test, y_pred, digits=4))
        
        fold += 1

    # Summary
    print("\n✅ Cross-Validation Summary:")
    print(f"Average Accuracy: {np.mean(accuracies):.4f}")
    print(f"Average F1 Score: {np.mean(f1s):.4f}")
    print(f"Average ROC AUC: {np.mean(aucs):.4f}")


In [46]:
# Runner Logistic Regression
featured_df = domain_featuring(raw_df)
base_model = LogisticRegression(max_iter=1000, solver='liblinear', class_weight='balanced', random_state=42)
print("Without Feature Engineering as is")
model_evaluation(base_model, raw_df)
print("Wth Domain Feature Engineering")
model_evaluation(base_model, featured_df)


Without Feature Engineering as is

🔹 Fold 1
Accuracy: 0.8632
F1 Score: 0.9120
ROC AUC: 0.9117
Confusion Matrix:
[[18389  5511]
 [10739 84160]]
              precision    recall  f1-score   support

         0.0     0.6313    0.7694    0.6936     23900
         1.0     0.9385    0.8868    0.9120     94899

    accuracy                         0.8632    118799
   macro avg     0.7849    0.8281    0.8028    118799
weighted avg     0.8767    0.8632    0.8680    118799


🔹 Fold 2
Accuracy: 0.8635
F1 Score: 0.9122
ROC AUC: 0.9112
Confusion Matrix:
[[18373  5527]
 [10685 84214]]
              precision    recall  f1-score   support

         0.0     0.6323    0.7687    0.6939     23900
         1.0     0.9384    0.8874    0.9122     94899

    accuracy                         0.8635    118799
   macro avg     0.7853    0.8281    0.8030    118799
weighted avg     0.8768    0.8635    0.8683    118799


🔹 Fold 3
Accuracy: 0.8612
F1 Score: 0.9106
ROC AUC: 0.9098
Confusion Matrix:
[[18335  5565]
 

KeyError: 'credit_utilization'

In [ ]:
# Model search for best params
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.linear_model import LogisticRegression

# Define the model
log_reg = LogisticRegression(solver='saga', max_iter=1000, random_state=42)

# Define parameter grid
param_grid = {
    'penalty': ['l1', 'l2', 'elasticnet'],
    'C': [0.01, 0.1, 1, 10],
    'class_weight': [None, 'balanced'],
    'l1_ratio': [0.0, 0.5, 1.0]  # only used when penalty='elasticnet'
}

# Define stratified 5-fold cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Grid search
grid_search = GridSearchCV(
    estimator=log_reg,
    param_grid=param_grid,
    scoring='roc_auc',     # or 'f1', depending on your goal
    cv=cv,
    n_jobs=-1,
    verbose=2
)

# Fit
grid_search.fit(X, y)

# Results
print("✅ Best Parameters:", grid_search.best_params_)
print("Best ROC AUC:", grid_search.best_score_)

# Final model (already retrained on all data with best params)
best_model = grid_search.best_estimator_


In [ ]:
# oof_preds = np.zeros(len(X))
# test_preds = np.zeros(len(test))

# for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
#     print(f'--- Fold {fold}/{N_SPLITS} ---')
    
#     X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
#     y_train, y_val = y.iloc[train_idx], y.iloc[val_idx] 
#     X_test = test[FEATURES].copy()

#     TE = TargetEncoder(cols_to_encode=INTER, cv=5, smooth='auto', aggs=['mean'], drop_original=True)
#     X_train = TE.fit_transform(X_train, y_train)
#     X_val = TE.transform(X_val)
#     X_test = TE.transform(X_test)

#     X_train[CATS] = X_train[CATS].astype('category')
#     X_val[CATS] = X_val[CATS].astype('category')
#     X_test[CATS] = X_test[CATS].astype('category')

#     model = LGBMClassifier(**params)
    
#     model.fit(X_train, y_train,
#               eval_set=[(X_val, y_val)])

#     val_preds = model.predict_proba(X_val)[:, 1]
#     oof_preds[val_idx] = val_preds
    
#     fold_score = roc_auc_score(y_val, val_preds)
#     print(f'Fold {fold} AUC: {fold_score:.4f}')
#     test_preds += model.predict_proba(X_test)[:, 1] / N_SPLITS

# overall_auc = roc_auc_score(y, oof_preds)
# print(f'====================')
# print(f'Overall OOF AUC: {overall_auc:.4f}')
# print(f'====================')